In [1]:
import numpy as np
import keras
import matplotlib.pyplot as plt

2026-04-21 13:32:26.703022: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776771146.759819   28363 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776771146.778745   28363 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-21 13:32:26.869294: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
from hgq.utils.sugar import Dataset

input_shape = (28, 28, 1)

# Load the data and split it between train and test sets
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# Binarize the images (convert to black and white) instead of scaling
X_train, X_test = (X_train > 127).astype(np.float32), (X_test > 127).astype(np.float32)

# Scale images to the [0, 1] range
#X_train = X_train.astype("float32") / 255
#X_test = X_test.astype("float32") / 255

#(Not required for HGQ)
# Make sure images have shape (28, 28, 1)
#X_train = np.expand_dims(X_train, -1)
#X_test = np.expand_dims(X_test, -1)
#print("X_train shape:", X_train.shape)
#print(X_train.shape[0], "train samples")
#print(X_test.shape[0], "test samples")

#(using SparseCategoricalCrossentropy instead)
# Convert class vectors to binary class matrices
#y_train = keras.utils.to_categorical(y_train, 10)
#y_test = keras.utils.to_categorical(y_test, 10)

#Shuffles the data
rng = np.random.default_rng(42)
order = rng.permutation(len(X_train))
X_train, y_train = X_train[order], y_train[order]

#Splits the training data into a training and validation set, equivalent to validation_split=0.1 in model.fit

N_train = int(0.9 * len(X_train))
X_train, X_val = X_train[:N_train], X_train[N_train:]
y_train, y_val = y_train[:N_train], y_train[N_train:]

# shuffle=True responsible for per Epoch variation
dataset_train = Dataset(X_train, y_train, batch_size=5000, device='gpu:0', shuffle=True)
dataset_val = Dataset(X_val, y_val, batch_size=5000, device='gpu:0')

I0000 00:00:1776767668.406688   42333 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5164 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
2026-04-21 12:34:29.127555: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 169344000 exceeds 10% of free system memory.
2026-04-21 12:34:33.292933: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 18816000 exceeds 10% of free system memory.


In [2]:
from keras.models import Sequential
from keras.optimizers import Adam
from keras.losses import SparseCategoricalCrossentropy
from hgq.config import QuantizerConfig, QuantizerConfigScope
from hgq.layers import QDense, QConv2D, QMaxPooling2D, QAveragePooling2D, QSoftmax, QEinsumDenseBatchnorm
from hgq.utils.sugar import FreeEBOPs, PBar

In [8]:
import keras
import numpy as np
from hgq.layers import QConv2D, QDense
from hgq.config import LayerConfigScope, QuantizerConfigScope

# First, set up quantization configuration
# For weights, use SAT_SYM overflow mode
with QuantizerConfigScope(q_type='kif', place='weight', overflow_mode='SAT_SYM', round_mode='RND'):
    # For activations, use different config
    with QuantizerConfigScope(q_type='kif', place='datalane', overflow_mode='WRAP', round_mode='RND'):
        with LayerConfigScope(enable_ebops=True, beta0=1e-5):
            # Create model with quantized layers
            model = keras.Sequential([
                keras.layers.Reshape((28, 28, 1)),
                keras.layers.MaxPooling2D((2, 2)),
                QConv2D(16, (3, 3), activation='relu'),
                keras.layers.MaxPooling2D((2, 2)),
                QConv2D(32, (3, 3), activation='relu'),
                keras.layers.MaxPooling2D((2, 2)),
                keras.layers.Flatten(),
                QDense(10)
            ])

# Compile model as usual
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape_1 (Reshape)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_conv2d_2 (QConv2D)            │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_conv2d_3 (QConv2D)            │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_dense_1 (QDense)              │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [3]:
# overflow modes:
#'sat_sym' clips to max value if overflow, 'wrap' wraps around (modulo) if overflow
scope_weights_and_biases = QuantizerConfigScope(
    place='all', 
    default_q_type='kbi', 
    heterogeneous_axis=None, 
    homogeneous_axis=(),
    overflow_mode='sat_sym'
)

scope_activations = QuantizerConfigScope(
    place='datalane', 
    default_q_type='kif', 
    heterogeneous_axis=None, 
    homogeneous_axis=(0, 1),
    overflow_mode='wrap'
)

In [14]:
def build_model(beta0=1e-5):
    with scope_weights_and_biases, scope_activations:

        model = keras.Sequential([
        
            keras.layers.Input(shape=(16,)), 
            
            QDense(64, beta0=beta0, activation='relu', name='dense_0'),
            QDense(32, beta0=beta0, activation='relu', name='dense_1'),
            QDense(32, beta0=beta0, activation='relu', name='dense_2'), 
            QDense(5, beta0=beta0, name='dense_3'),
        ])
    return model

model = build_model(beta0=0.5e-5)
loss = keras.losses.CategoricalCrossentropy(from_logits=True)
opt = keras.optimizers.Adam(learning_rate=5e-3)

model.compile(opt, loss, metrics=['accuracy'], jit_compile=True, steps_per_execution=4)

In [15]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_0 (QDense)                │ (None, 64)             │         4,358 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (QDense)                │ (None, 32)             │         8,326 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (QDense)                │ (None, 32)             │         4,230 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (QDense)                │ (None, 5)              │           666 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,580 (55.80 KB)

 Trainable params: 13,171 (51.45 KB)

 Non-trainable params: 4,409 (4.35 KB)

In [4]:
from hgq.utils.sugar import Dataset

input_shape = (28, 28, 1)

# Load the data and split it between train and test sets
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# Scale images to the [0, 1] range
X_train = X_train.astype("float32") / 255
X_test = X_test.astype("float32") / 255

# Make sure images have shape (28, 28, 1)
X_train = np.expand_dims(X_train, -1)
X_test = np.expand_dims(X_test, -1)
print("X_train shape:", X_train.shape)
print(X_train.shape[0], "train samples")
print(X_test.shape[0], "test samples")

N_train = int(0.9 * len(X_train))
X_train, X_val = X_train[:N_train], X_train[N_train:]
y_train, y_val = y_train[:N_train], y_train[N_train:]
# Convert class vectors to binary class matrices
y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test, 10)
y_val = keras.utils.to_categorical(y_val, 10)

dataset_train = Dataset(X_train, y_train, batch_size=5000, device='gpu:0', shuffle=True)
dataset_val = Dataset(X_val, y_val, batch_size=5000, device='gpu:0')

X_train shape: (60000, 28, 28, 1)
60000 train samples
10000 test samples


I0000 00:00:1776771191.511686   28363 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5560 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
2026-04-21 13:33:11.543517: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 169344000 exceeds 10% of free system memory.


In [5]:
import keras
import numpy as np
from hgq.layers import QConv2D, QDense
from hgq.config import LayerConfigScope, QuantizerConfigScope

def build_model(beta0=1e-6):
    # 1. Define the input quantizer explicitly 
    iqc = QuantizerConfig(trainable=False, i0=1, k0=0, f0=0)
    
    with scope_weights_and_biases, scope_activations:
        model = keras.Sequential([
                keras.Input(shape=input_shape),
                QConv2D(32, (3, 3), activation='relu'),
                keras.layers.MaxPooling2D((2, 2)),
                QConv2D(32, (3, 3), activation='relu'),
                keras.layers.MaxPooling2D((2, 2)),
                keras.layers.Flatten(),
                keras.layers.Dropout(0.4),
                QDense(10)
            ])
    return model
model = build_model(beta0=1e-7)
loss = keras.losses.CategoricalCrossentropy(from_logits=True)
opt = keras.optimizers.Adam(learning_rate=0.001)

model.summary()

model.compile(optimizer=opt, loss=loss, metrics=['accuracy'], jit_compile=True, steps_per_execution=8)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ q_conv2d (QConv2D)              │ (None, 26, 26, 32)     │         1,367 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_conv2d_1 (QConv2D)            │ (None, 11, 11, 32)     │        38,243 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 800)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 800)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_dense (QDense)                │ (None, 10)             │        32,046 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 71,656 (227.10 KB)

 Trainable params: 53,179 (207.73 KB)

 Non-trainable params: 18,477 (19.37 KB)

In [13]:
input_shape = (28, 28, 1)
def build_model(beta0=1e-6):
    with scope_weights_and_biases, scope_activations:

        iqc = QuantizerConfig(trainable=False, i0=1, k0=0, f0=0)
        inp = keras.Input(shape=input_shape)
        out = QAveragePooling2D(pool_size=2, padding='valid', iq_conf=iqc)(inp)
        out = keras.layers.Flatten()(out)
        out = QEinsumDenseBatchnorm(
            'bc,cC->bC', 24, name='t1', bias_axes='C', activation='relu', kernel_regularizer=keras.regularizers.L2(1e-3)
        )(out)
        out = QEinsumDenseBatchnorm(
            'bc,cC->bC', 24, name='t2', bias_axes='C', activation='relu', kernel_regularizer=keras.regularizers.L2(1e-3)
        )(out)
        out = QEinsumDenseBatchnorm('bc,cC->bC', 24, name='t3', bias_axes='C', activation='relu')(out)
        out = QDense(10, name='out')(out)
    return keras.Model(inputs=inp, outputs=out)

model = build_model(beta0=1e-6)
loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
opt = keras.optimizers.Adam(learning_rate=0.001)

model.summary()

model.compile(optimizer=opt, loss=loss, metrics=['accuracy'], jit_compile=True, steps_per_execution=8)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_average_pooling2d             │ (None, 14, 14, 1)      │            87 │
│ (QAveragePooling2D)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 196)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ t1 (QEinsumDenseBatchnorm)      │ (None, 24)             │        18,990 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ t2 (QEinsumDenseBatchnorm)      │ (None, 24)             │         2,478 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ t3 (QEinsumDenseBatchnorm)      │ (None, 24)             │         2,478 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ out (QDense)                    │ (None, 10)             │         1,006 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,039 (79.62 KB)

 Trainable params: 18,610 (72.70 KB)

 Non-trainable params: 6,429 (6.92 KB)

In [6]:
pbar = PBar('loss: {loss:.3f}/{val_loss:.3f} - acc: {accuracy:.3f}/{val_accuracy:.3f}')


ebops = FreeEBOPs()
nan_terminate = keras.callbacks.TerminateOnNaN()
callbacks = [ebops, pbar, nan_terminate]

In [7]:
model.fit(dataset_train, epochs=200, validation_data=dataset_val, callbacks=callbacks, verbose=0)

  0%|          | 0/200 [00:00<?, ?epoch/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1776771223.632957   28772 service.cc:148] XLA service 0x78fa7c005e00 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776771223.633013   28772 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2026-04-21 13:33:43.755011: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1776771224.203807   28772 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-04-21 13:33:44.870952: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_2088', 8 bytes spill stores, 8 bytes spill loads

2026-04-21 13:33:44.917625: I external/local_xla/xla/stream_

In [10]:
score = model.evaluate(X_test, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])

Test loss: 0.3598361015319824
Test accuracy: 0.8913999795913696


In [ ]:
model.save('/home/slopin/DAT255-project/Modeller/mnist_baseline.keras')